In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn"]
ASSETS = [{"name": "fitness analysis.csv", "url": "/labs/2.2.5/fitness%20analysis.csv", "size": 149669, "sha256": "d1b18b5115bc4ef833b6cc00c4bda1bf5223c65c1ec5f3ace0cc8900e00e28e3"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.2.5 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 加载数据集
df = __________

# 显示前五行数据
print(__________)

# 选择相关特征进行建模
X = df[['Your gender ', 'How important is exercise to you ?', 'How healthy do you consider yourself?']]
X = __________(X)  # 将分类变量转为数值变量

# 设置目标变量
y = __________  

# 将数据集划分为训练集和测试集（测试集占20%）
X_train, X_test, y_train, y_test = __________(__________, random_state=42)

# 创建并训练决策树回归模型
__________ = __________(random_state=42)
# 训练决策树回归模型
__________

# 保存训练好的模型
with open('2.2.5_model.pkl', 'wb') as model_file:
    pickle.__________

# 进行预测
y_pred = __________

# 将结果保存到文本文件中
results = pd.DataFrame({'实际值': y_test, '预测值': y_pred})
results_filename = '2.2.5_results.txt'
__________(__________, index=False, sep='\t')  

# 将测试结果保存到报告文件中
report_filename = '2.2.5_report.txt'
with open(__________) as f:
    f.write(f'均方误差: {__________}\n')
    f.write(f'平均绝对误差: {__________}\n')
    f.write(f'决定系数: {__________}\n')